This notebook explores and analyses my YouTube watch history using Python and pandas. It covers parsing raw Google Takeout data, cleaning and structuring watch records, and performing exploratory data analysis with a focus on music listening habits

# Parsing Watch History

## Initiating watch history retrieval

In [4]:
from enum import unique
from pathlib import Path
from bs4 import BeautifulSoup
from pandas.errors import CategoricalConversionWarning

project_root = Path.cwd().parent
watch_path = project_root / "data" / "raw" / "watch-history.html"

print(watch_path)
print(watch_path.exists())

C:\Users\MsnSi\PycharmProjects\Youtube_Music_Analysis\data\raw\watch-history.html
True


## Read data from watch history

Beautiful Soup is a Python library for parsing HTML/XML and turning it into something Python can navigate. Line 1 (htm) loads the html file into one string, and line 2 (soup) is used to parse through that string.


cards
│
├── cards[0] ── Watch history entry #1
├── cards[1] ── Watch history entry #2
├── cards[2] ── Watch history entry #3
├── cards[3] ── Watch history entry #4
└── ...

In [5]:
html = watch_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "lxml")

cards = soup.select("div.outer-cell")

print(f"Number of activity records: {len(cards)}")

Number of activity records: 12200


## Grabbing the data from watch history


In [6]:
first_card = cards[0]
content = first_card.select_one("div.content-cell")

list(content.stripped_strings)

['Watched',
 'Mario Kart 8 Deluxe Players are Going Insane right now...',
 'Shortcat',
 '7 Sept 2026, 13:51:45 IST']

## Converting the Watched Video Data records into a dictionary data structure

In [7]:
parts = list(content.stripped_strings)
links = content.select("a")


record = {
    "action": parts[0],
    "title": parts[1],
    "channel": parts[2],
    "watched_at": parts[3],
    "video_url": links[0].get("href"),
}

print(record)

{'action': 'Watched', 'title': 'Mario Kart 8 Deluxe Players are Going Insane right now...', 'channel': 'Shortcat', 'watched_at': '7 Sept 2026, 13:51:45 IST', 'video_url': 'https://www.youtube.com/watch?v=TOKdln0vkrw'}


In [8]:
## Creating all the Records from Watch history

Create empty records list

FOR every card:
    Find the content inside this card

    Extract the strings
    Extract the links

    Build a dictionary

    Add dictionary to records

Finished → records now contains every watch-history entry

In [15]:
records = []
record_number = 0
bad_records = 0

for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if len(parts) < 4:
        bad_records += 1
        print("PARTS:", parts)
        print("LINKS:", links)
        print("--------------------")
    else:
        record_number += 1

    #record = {
    #    "action": parts[0],
    #    "title": parts[1],
     #   "channel": parts[2],
      #  "watched_at": parts[3],
       # "video_url": links[0].get("href"),
    #}

    #records.append(record)
print("Number of normal records", record_number)
print("Number of bad records: ", bad_records)


PARTS: ['Watched', 'INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1', '4 Sept 2026, 21:23:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=J5wkIKQfW_I">INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1</a>]
--------------------
PARTS: ['Watched', 'INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO', '3 Sept 2026, 23:15:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=KkFMhhuYruM">INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO</a>]
--------------------
PARTS: ['Viewed a post that is no longer available', '1 Sept 2026, 03:39:29 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '31 Aug 2026, 14:24:55 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '29 Aug 2026, 22:38:37 IST']
LINKS: []
--------------------
PARTS: ['Watched', 'https://www.youtube.com/watch

It seems like our parser is a bit flawed due to some bad data within the watch history, there seems to be 4 types of outliers:

1. Ads: 3 Parts, but no obvious link
2. Community Posts: 3 parts, no link
3. Unlisted Videos: 3 Parts, Link exists in the *links* field
4. Privated Videos: 3 Parts, 1 Link in parts, 2 links in *links field


## Updated Record Retrieval


In [40]:
records = []
record_number = 0
missing_channel_records = 0
non_video_records = 0

zero_links = 0
one_link = 0
two_or_more_links = 0


for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if parts[0] != 'Watched':
        non_video_records += 1
        continue

    if len(parts) == 4:
        record_number += 1
        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": parts[2],
            "watched_at": parts[3],
            "video_url": links[0].get("href"),
        }

    elif len(parts) == 3:
        missing_channel_records += 1

        if len(links) <= 0:
            zero_links += 1
        elif len(links) == 1:
            one_link += 1
        elif len(links) == 2:
            two_or_more_links += 1

        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": None,
            "watched_at": parts[2],
            "video_url": links[0].get("href"),
        }


    records.append(record)

print("Number of normal records", record_number)
print("Number of records with missing channels: ", missing_channel_records)
print("Number of non-video records: ", non_video_records)

print("Number of bad records with zero links: ", zero_links)
print("Number of records with one link: ", one_link)
print("Number of records with two or more links: ", two_or_more_links)

Number of normal records 9945
Number of records with missing channels:  115
Number of non-video records:  2140
Number of bad records with zero links:  0
Number of records with one link:  115
Number of records with two or more links:  0


In [41]:
zero_links = 0
one_link = 0
two_or_more_links = 0

# Using Pandas

## Implementing the Pandas Library

In [42]:
import pandas as pd

records_dataframe = pd.DataFrame(records)

records_dataframe.head()

,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,"7 Sept 2026, 13:51:45 IST",https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,"7 Sept 2026, 13:50:46 IST",https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,"7 Sept 2026, 13:44:34 IST",https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,"7 Sept 2026, 13:43:43 IST",https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,"7 Sept 2026, 13:35:14 IST",https://www.youtube.com/watch?v=Ftji2LKqVXs


## Exploring the properties of the DataFrame

We can use the shape attribute to understand the general structure of the DataFrame

In [43]:
records_dataframe.shape


(10060, 5)

in the below code, we get (100060,5) which likely represents the number of row entries and the number of coloumn entries in our DataFrame

The dtypes attribute is used to expose the types of each column entry in the dataFrame

In [44]:
records_dataframe.dtypes


action        str
title         str
channel       str
watched_at    str
video_url     str
dtype: object

Before changing the data, We already know there should be at least 115 missing channel values from our parsing work, we need to figure out what's missing from those entries.

We can use the isna() function, which will scan through each cell in the Data Frame and check if a value exists, combine this with .sum() to add the up these entries together to see how many of each value is missing.

Previously both channel and video_url had shown 115 entries, this was a mistake as a result of a flawed parser which failed to catch the fact that I failed to realise that just because an entry might have less parts (due to not having a channel), we could still grab their link as that was from another array from the content entry

In [46]:
records_dataframe.isna().sum()

action          0
title           0
channel       115
watched_at      0
video_url       0
dtype: int64

# Initial Data Cleaning

We need to clean up the data so we can use it for analysis, firstly we'll convert the "watched_at" field into a timestamp so it can be used later for analysis

## Getting Rid of Bad Entries

Getting Rid of Bad Entries

In [90]:
valid_channel_records = records_dataframe[records_dataframe["channel"].notna()]

records_dataframe = valid_channel_records


#print(records_dataframe.shape)
#records_dataframe.head()


records_dataframe.reset_index(drop = True)

,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs
...,...,...,...,...,...
9940,Watched,Wonder Acute (Onsen) #ウマ娘 #umamusume #anime #...,Kidd Thunder 76,2026-07-07 17:56:08,https://www.youtube.com/watch?v=fbnlwV1jmdE
9941,Watched,I NEVER lost faith in YukaF,NiceWigg,2026-07-07 17:54:06,https://www.youtube.com/watch?v=BcbTEjpaY5M
9942,Watched,Haru Breaks The Fourth Wall #voiceacting#voice...,hapyr,2026-07-07 17:54:00,https://www.youtube.com/watch?v=qXvvSKL0hgA
9943,Watched,An Agnes Digital Interaction #animecharacter#a...,hapyr,2026-07-07 17:53:34,https://www.youtube.com/watch?v=wUfEL06bVSM


We can use the .str attribute to access all the strings in a specific row entry, and then we can use [n:n] to slice off a specific section of the string. In this context, we slice off the timezone so we can more easily convert our entry times that are currently strings into timestamp data types

In [92]:
#converted_watched_at_entries = pd.to_datetime(records_dataframe["watched_at"].str[:-4], dayfirst=True)

#records_dataframe["watched_at"] = converted_watched_at_entries

records_dataframe.head()



,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs


## Determining what is a music video or not

There are a few methods we could use to track whether a video is music related or not.
- Keeping track of a channel that is music-focused and checking if a video comes from that channel (also works for topic channels
- Checking the title of the video for certain keywords
- Using playlist data, if a video appears in a playlist, then it should be included
- Checking the metadata from the URL, either from the Takeout data or Youtube's own metadata

To do that though, we'll need to extend our DataFrame furhter to be able to include video ID as a metric for comparison

In [99]:
video_ids = records_dataframe["video_url"].str.split("v=").str[1]

video_ids.str.len().unique() #Checking that all video ids equally return 11 characters

records_dataframe["video_ids"] = video_ids

records_dataframe.head()
#records_dataframe.shape

,action,title,channel,watched_at,video_url,video_ids
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,2026-09-07 13:51:45,https://www.youtube.com/watch?v=TOKdln0vkrw,TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,2026-09-07 13:50:46,https://www.youtube.com/watch?v=Lg1QI8ZkPRU,Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,2026-09-07 13:44:34,https://www.youtube.com/watch?v=5OOE4WORb5Q,5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,2026-09-07 13:43:43,https://www.youtube.com/watch?v=R9922GyZmvw,R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,2026-09-07 13:35:14,https://www.youtube.com/watch?v=Ftji2LKqVXs,Ftji2LKqVXs


### Reading CSV Files for Video IDs

Now we want to try converting the CSV files from the music and playlist folders from the google takeout data to act as our first basis for finding music videos.

In [100]:
music_library_csv_path = project_root / "data" / "raw" / "music library songs.csv"

music_library_dataframe = pd.read_csv(music_library_csv_path)


In [101]:
music_library_dataframe.head()


,Video ID,Song Title,Album Title,Artist Name 1,Artist Name 2
0,1aTIvok5__w,Across the Galaxies,Deep Sleep Relaxation Music,Mrm Team,NaN
1,B2i_23Ufnj0,aLIEz [mZk ver.] - aLIEz (mZk Version),&Z,SawanoHiroyuki[nZk],mizuki
2,tQRk5OFsGXo,Andromeda,Still Still Stellar,星街すいせい,NaN
3,UxG0N1Nfoq0,朝です… - Asadesu,Seishunbutayarouha Bunny Girl Senpaino Yumeomi...,fox capture plan,NaN
4,fFB88ikHOGI,Aurora: Delta Waves 2.5 Hz,"Brain Waves Music, Vol. 1 - Delta Waves",Mrm team,NaN


Now that we have a 2nd dataframe, we can start checking if there are songs in our watch history that are listed within some of our music libraries

In [102]:
library_matches = music_library_dataframe["Video ID"].isin(records_dataframe["video_ids"])

library_matches.sum()

np.int64(2)

So it seems like there are 2 instances of music in the library that were in my watch history.

While i think that this method is interesting, I'm unsure how useful it could be, at least for this specific instance of my Google takeout as it is very small and relatively recent (only going back to the 7th of July this year), which would mean cross-referencing video ids to playlists would be relatively pointless given that I've mostly switched to usingYyoutube's auto generated playlists, and that I also don't listen to those songs as often (if at all) anymore. in my opinion, I think it would be a better idea to use some of our other methods.

# Using Youtube's Data API

## Basics of receiving requests

Firstly, we want to import our Youtube API Key

In [103]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("YOUTUBE_API_KEY")
print(api_key is not None)



True


Now let's make an API request using our DataFrame

In [136]:
#records_dataframe.head(170)

music_video_id = records_dataframe.loc[168, "video_ids"]
music_video_title = records_dataframe.loc[168,"title"]

non_music_video_id = records_dataframe.loc[26, "video_ids"]
non_music_video_title = records_dataframe.loc[26, "title"]

print(f'The video: {music_video_title}, has the video id: {music_video_id}')
print(f'The non music video: {non_music_video_title}, has the video id: {non_music_video_id}')

The video: Why Do I (with Hatsune Miku), has the video id: HMFNXgu7L3Q
The non music video: MAFIA 3 DEFINITIVE EDITION All Cutscenes (Full Game Movie) 4K Ultra HD, has the video id: qNdhM8j7scU


In [137]:
import requests

url = "https://www.googleapis.com/youtube/v3/videos"

params = {
    "part": "snippet,contentDetails",
    "id": music_video_id,
    "key": api_key
}

non_music_video_params = {
    "part": "snippet,contentDetails",
    "id": non_music_video_id,
    "key": api_key
}

music_video_response = requests.get(url, params=params)
non_music_video_response = requests.get(url, params=non_music_video_params)

music_video_data = music_video_response.json()
non_music_video_data = non_music_video_response.json()
#print(data)

In [138]:
mv_item = music_video_data["items"][0]
nmv_item = non_music_video_data["items"][0]

print(mv_item.keys())
print(mv_item["snippet"].keys())
print(mv_item["contentDetails"].keys())

dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails'])
dict_keys(['publishedAt', 'channelId', 'title', 'description', 'thumbnails', 'channelTitle', 'tags', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'localized', 'defaultAudioLanguage'])
dict_keys(['duration', 'dimension', 'definition', 'caption', 'licensedContent', 'regionRestriction', 'contentRating', 'projection'])


In [139]:
mv_snippet = mv_item["snippet"]
mv_content = mv_item["contentDetails"]

print("Title:", mv_snippet["title"])
print("Channel:", mv_snippet["channelTitle"])
print("Category:", mv_snippet["categoryId"])
print("Duration:", mv_content["duration"])
print("Tags:", mv_snippet.get("tags", []))

print("")
nmv_snippet = nmv_item["snippet"]
nmv_content = nmv_item["contentDetails"]

print("Title:", nmv_snippet["title"])
print("Channel:", nmv_snippet["channelTitle"])
print("Category:", nmv_snippet["categoryId"])
print("Duration:", nmv_content["duration"])
print("Tags:", nmv_snippet.get("tags", []))

Title: Why Do I (with Hatsune Miku)
Channel: Set It Off
Category: 10
Duration: PT2M36S
Tags: ['set it off', 'upside down', 'cody carson', 'maxx danziger', 'zach dewall', 'dan clermont', 'life afraid', 'hypnotized', 'wolf in sheeps clothing', "wolf in sheep's clothing", 'Duality', 'BAND VLOG', 'BAND VLOGS', 'why worry', 'bleak December', 'ancient history', 'Forever stuck in our youth', 'SIO', '21 Pilots', 'Twenty one Pilots', 'panic at the disco', 'brendon urie', 'top', 'Joshdun', 'MCR', 'My chemical romance', 'pete wentz', 'Fan girl', 'Set it off band', 'All time Low', 'night core', 'nightcore']

Title: MAFIA 3 DEFINITIVE EDITION All Cutscenes (Full Game Movie) 4K Ultra HD
Channel: Gamer's Little Playground
Category: 20
Duration: PT6H3M40S
Tags: ['game', 'full', 'gameplay', 'no commentary', 'full gameplay', 'MAFIA 3 DEFINITIVE EDITION All Cutscenes (Full Game Movie) 4K Ultra HD', 'mafia 3 full game', 'full gameplay walkthrough', 'mafia 3 definitive edition', 'mafia 3', 'mafia 3 all cut

## Finding the nuances between "music videos"

We now need to find an example of videos that may be considered on the cusp of being a music video

- a music video uploaded by a non-official channel: Umamusume] yume wo kakeru! (Canopus Ver) (Lyrics/Color Coded) [TV Anime 2nd season https://www.youtube.com/watch?v=OpbokSpFAPk&list=RDOpbokSpFAPk&start_radio=1
- a cover/remix/nightcore song: IE 奏 - スキマスイッチ (Cover) / VESPERBELL ヨミ
https://www.youtube.com/watch?v=-BYdGfsSrHA&list=RD-BYdGfsSrHA&start_radio=1
- a live performance : Hana ni Bourei (Yorushika) - Inui Toko x Hoshimachi Suisei [ Lyrics / Nijisanji / Hololive ] - https://www.youtube.com/watch?v=2zh66iI3fVc&list=RD2zh66iI3fVc&start_radio=1
- a normal ~3–5 minute non-music video; Gus Fring's Astronomical Offer : https://www.youtube.com/watch?v=ThZMClhU35I&t=1s
- a very long music mix/album - [1 Hour] Yorushika ヨルシカ songs collection / playlist https://www.youtube.com/watch?v=nOeWWduMb9Q&list=RDnOeWWduMb9Q&start_radio=1
- a video that contains music but that you personally wouldn't consider a music listen: Popular Songs That Were Later Used in Anime - https://www.youtube.com/watch?v=by2Dx-_souY

In [140]:
non_official_music = "OpbokSpFAPk"
cover = "-BYdGfsSrHA"
live_performance = "2zh66iI3fVc"
short_non_music = "ThZMClhU35I"
long_music = "nOeWWduMb9Q"
contains_music = "by2Dx-_souY"

test_video_ids = [ non_official_music, cover, live_performance, short_non_music, long_music, contains_music ]

combined_video_ids = ",".join(test_video_ids)

test_params = {
    "part": "snippet,contentDetails",
    "id": combined_video_ids,
    "key": api_key
}

test_response = requests.get(url, params=test_params)
test_data = test_response.json()

for item in test_data["items"]:
    test_snippet = item["snippet"]
    test_content = item["contentDetails"]

    print("Title:", test_snippet["title"])
    print("Channel:", test_snippet["channelTitle"])
    print("Category:", test_snippet["categoryId"])
    print("Duration:", test_content["duration"])
    print("Tags:", test_snippet.get("tags", []))
    print("")


Title: 【ウマ娘】ユメヲカケル! (カノープスVer) (パート分け/Color Coded/Lyrics)【Run for Our Dream】
Channel: 🐈Kogane Music
Category: 10
Duration: PT4M52S
Tags: ['ウマ娘', 'パート分け', 'ユメヲカケル!', '賽馬娘', 'yumewokakeru', 'yume wo kakeru', '讓夢想成真吧', '让梦想成真吧', 'パート割', '歌割り', 'lyrics', 'kogane music', 'フル', 'full', '우마무스메']

Title: 奏  - スキマスイッチ (Cover) / VESPERBELL ヨミ
Channel: VESPERBELL_YM
Category: 10
Duration: PT5M30S
Tags: ['VESPERBELL', '歌ってみた', 'カバー', 'cover', 'VTuber', 'バーチャルYouTuber', 'ヴェスパーベル', 'YOMI', 'KASUKA', 'ヨミ', 'カスカ', 'Vsinger']

Title: Hana ni Bourei (Yorushika) - Inui Toko x Hoshimachi Suisei [ Lyrics / Nijisanji / Hololive ]
Channel: Yudha - ユダ
Category: 10
Duration: PT4M34S
Tags: ['Nijisanji', 'Vtuber', 'Virtual Youtuber', 'Vtuber ENG', 'Nijisanji ENG', 'Hololive', 'Inui Toko', 'Hoshimachi Suisei', 'Inui', 'Suisei', 'Inui Eng', 'Suisei Eng', 'Hana ni Bourei']

Title: Gus Fring's Astronomical Offer | No Más | Breaking Bad
Channel: Breaking Bad & Better Call Saul
Category: 1
Duration: PT3M23S
Tags: ['Br

Interesting to see is that the ones that I would've considered music were also agreed to be by Youtube's metadata, and the ones that weren't like the non music video was not, interestingly, Youtube was agreed with my own opinion about the last 2 videos, the music compilation was considered a music video (which by in large I would mostly agree with) and the popular songs video which definitely might've tripped up people wasn't strictly considered a music video, so this is definitely a good indicator we should use.

# Further Data Cleaning

## Filtering the Watch history

Firstly, we'll need to combine video ids together in order to reduce the amount of API requests we're calling

In [144]:
start_index = 0
BATCH_SIZE = 50

combined_video_ids_batch = []

for start_index in range(start_index, len(records_dataframe), BATCH_SIZE):
    video_ids_batch = records_dataframe["video_ids"][start_index:start_index+BATCH_SIZE]
    combined_video_ids = ",".join(video_ids_batch)
    combined_video_ids_batch.append(combined_video_ids)

print(len(combined_video_ids_batch))

199


In [145]:
music_video_total = 0
for combined_video_ids_string in combined_video_ids_batch:
    watch_history_params = {
        "part": "snippet,contentDetails",
        "id": combined_video_ids_string,
        "key": api_key
    }

    watch_history_response = requests.get(url, params=watch_history_params)
    watch_history_data = watch_history_response.json()

    for item in watch_history_data["items"]:
        watch_history_snippet = item["snippet"]
        watch_history_content = item["contentDetails"]

        video_title = watch_history_snippet["title"]
        video_category = watch_history_snippet["categoryId"]

        if (video_category == "10"):
            print(f"{video_title} is considered a music video by YouTube's metadata")
            music_video_total += 1

print(f"Total music videos: {music_video_total}")



Unknown Mother Goose (wowaka) ♡ English Cover【rachie】アンノウン・マザーグース is considered a music video by YouTube's metadata
Aldnoah.Zero - "aLIEz" (FULL Ending) | ENGLISH ver | AmaLee is considered a music video by YouTube's metadata
"Violet Snow" (Violet Evergarden) 🍓 Cover by Lizz Robinett ft. The Goatee is considered a music video by YouTube's metadata
Nightcore / Sped Up → Ai Đưa Em Về (Rock Version/Cover) is considered a music video by YouTube's metadata
monitoring (deco*27) ♥ english cover【rachie】モニタリング is considered a music video by YouTube's metadata
GUM&DROP / 星街すいせい (official) is considered a music video by YouTube's metadata
The Vampire (DECO*27) ♥ English Cover 【rachie】ヴァンパイア is considered a music video by YouTube's metadata
稲葉曇『ラグトレイン』Vo. 歌愛ユキ is considered a music video by YouTube's metadata
Official髭男dism - Pretender［Official Video］ is considered a music video by YouTube's metadata
【Ado】ギラギラ is considered a music video by YouTube's metadata
Midnight Grand Orchestra『Moonlightspee

Ok, so of the 9945 watch history entries, only 630 songs that were considered songs by Youtube's metadata, which for being my main method of listening to music, is honestly quite low (that being said, I'd have to look deeper into the data, and additionally, the Hong Kong Japan trip likely would've put a damper on the data, If I Had done this say right now while in the semester, I'd say the ratio would be a lot better, it also doesn't help that Youtube changed their watch history logic such that any passing view counts as a view (in the case for Youtube statistics) which may be reflected in my own watch history, further diluting the pool. Fortunately, the process was incredibly quick at just 2 minutes and 3 seconds, which makes me interested if I were to take a longer range of time, how it would look.

Unfortunately, there is another issue, that being bad data, mainly false positives, you could break them up half and half into shorts and actual full videos.

With the shorts, my guess is that the ones that are either tagged as music or have a music artist as the subject were likley flagged as music, then there are the meme compilations by MrUnknown and the TV/Movie clips which confuse me.

As for the long form videos, they're a lot less dicpherable, some that are fully just music but whom i wouldn't necessarily consider its main entertainment subject, Sound Effects are particularly confusing, and then ASMR.....


## Shorts Classification

### Using Youtube's API to identify shorts

One of the major outliers I found checking the vidoes classified as music were Shorts, so an easy way to filter items would be to check if they are Shorts or not, and then if they aren't further checking before defining as music.

In [152]:
#The first 4 are the 4 most recent shorts in my watch history, the last 2 are definitively considered false positives

short_video_ids = ["TOKdln0vkrw", "Lg1QI8ZkPRU", "5OOE4WORb5Q", "R9922GyZmvw", "dq_4aV0uLus", "hgHR7AG3r6o", "p-BiFXlK268", "mrIYgy-xbyU", "K0mzD4i8AkI", "lV1RsohlBUc", "H66SAz682rc", "R7IfxRVT6ZQ", "mrIYgy-xbyU", "LY0p6VmRXbs"]

short_video_id = ",".join(short_video_ids)

short_params = {
    "part": "snippet,contentDetails",
    "id": short_video_id,
    "key": api_key
}

short_response = requests.get(url, params=short_params)
short_data = short_response.json()

for item in short_data["items"]:
    short_snippet = item["snippet"]
    short_content = item["contentDetails"]

    print("Title:", short_snippet["title"])
    print("Channel:", short_snippet["channelTitle"])
    print("Category:", short_snippet["categoryId"])
    print("Duration:", short_content["duration"])
    print("Tags:", short_snippet.get("tags", []))
    print("")



Title: Mario Kart 8 Deluxe Players are Going Insane right now...
Channel: Shortcat
Category: 20
Duration: PT1M49S
Tags: []

Title: Oops! Nintendo reveals Mario Kart 8's DEBUG mode
Channel: Beta64
Category: 20
Duration: PT58S
Tags: ['beta 64', 'beta64', 'beta', 'video game history', 'documentary', 'lost media', 'proto', 'prototype', 'beta build', 'trailer', 'early', 'unused', 'scrapped', 'interview', 'mario kart 8', 'mario kart 8 deluxe', 'wii u', 'switch', 'mario kart world', 'mario kart', 'kiosk demo', 'mario kart 8 demo', 'mario kart 8 beta', 'nintendo direct', 'mario kart 8 deluxe kiosk demo', 'debug mode', 'mario kart 8 debug']

Title: I Love These Uma Musume
Channel: Starfang
Category: 20
Duration: PT16S
Tags: []

Title: 第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #gunpla #bandai #高達 #toys #萬代
Channel: 玩具測評總動員
Category: 24
Duration: PT31S
Tags: []

Title: Natural selection 🥀
Channel: MrUnknown
Category: 10
Duration: PT15S
Tags: []

Title: He noticed the actor’s lines slowed down 

Unfortunately, there isnt any other data that we can use to identify a common behaviour (aside from duration, which at this point, feels a little dubious to use as our smoking gun), however there might be another avenue to use:

YouTube itself knows whether a video is a Short because the website routes Shorts through its /shorts/VIDEO_ID interface.

So instead of asking the Data API whether something is a Short, we may be able to ask YouTube's public web interface indirectly.
Conceptually:
video ID
   │
   ├── Data API ──────→ metadata/music classification
   │
   └── YouTube web ───→ does YouTube treat this ID as a Short?

In [158]:
known_short = "hgHR7AG3r6o"       # House clip, 2m52s
known_long = "oMuyO2tFDwg"              # Nicewigg ALGS video, 1h36m57s

short_url = f"https://www.youtube.com/shorts/{known_short}"
long_url = f"https://www.youtube.com/shorts/{known_long}"

short_response = requests.get(short_url)
long_response = requests.get(long_url)

print("URLs before being sent to Youtube API, notice how we sending both links to be viewed via the shorts player, even though we know one of them is a long form video\n")
print(short_url)
print(long_url)


print("\nSHORT:")
print(short_response.status_code)
print(short_response.url)
print("SHORT HISTORY:")
print(short_response.history)

print("\nLONG:")
print(long_response.status_code)
print(long_response.url)
print("LONG HISTORY:")
print(long_response.history)




URLs before being sent to Youtube API, notice how we sending both links to be viewed via the shorts player, even though we know one of them is a long form video

https://www.youtube.com/shorts/hgHR7AG3r6o
https://www.youtube.com/shorts/oMuyO2tFDwg
SHORT:
200
https://www.youtube.com/shorts/hgHR7AG3r6o
SHORT HISTORY:
[]

LONG:
200
https://www.youtube.com/watch?v=oMuyO2tFDwg&pp=0gcJCW8CQVKahPAF
LONG HISTORY:
[<Response [303]>]


In the code above, We tried to force a long form video to be viewed as a short, but it redirected back to the long form video. This should make up the basic function that we can use to easily determine if a video is a short, if we force it into a short, and it redirects as a short, we can classify it as a short and hence then cut it out from our history.

In [162]:
def is_short(video_id):
    video_url = f"https://www.youtube.com/shorts/{video_id}"
    response = requests.get(video_url)

    if "/shorts/" in response.url:
        return True

    #print(response.status_code)
    #print(response.url)
    return False

known_short_video_ids = ["dq_4aV0uLus", "hgHR7AG3r6o", "p-BiFXlK268", "mrIYgy-xbyU", "K0mzD4i8AkI", "lV1RsohlBUc", "H66SAz682rc", "R7IfxRVT6ZQ", "mrIYgy-xbyU"] #All videos are considred music at this stage
known_long_video_ids = ["uRDuw9BpnmY", "tab1yUu20gA", "oMuyO2tFDwg", "qQ6oJR6Ki9s", "UTcwnCz4-UY", "xnzfxv9J1KE", "4h0Ht6JyCxc", "RlDZgZvIQ5U"] #5 videos not considred music, 5 considered music

for known_short_video_id in known_short_video_ids:
    if is_short(known_short_video_id):
        print(f"{known_short_video_id} is a short video")
    else:
        print(f"{known_short_video_id} is not a short video")

for known_long_video_id in known_long_video_ids:
    if is_short(known_long_video_id):
        print(f"{known_long_video_id} is not a long video, it's a short video")
    else:
        print(f"{known_long_video_id} is a long video")





dq_4aV0uLus is a short video
hgHR7AG3r6o is a short video
p-BiFXlK268 is a short video
mrIYgy-xbyU is a short video
K0mzD4i8AkI is a short video
lV1RsohlBUc is a short video
H66SAz682rc is a short video
R7IfxRVT6ZQ is a short video
mrIYgy-xbyU is a short video
uRDuw9BpnmY is a long video
tab1yUu20gA is a long video
oMuyO2tFDwg is a long video
qQ6oJR6Ki9s is a long video
UTcwnCz4-UY is a long video
xnzfxv9J1KE is a long video
4h0Ht6JyCxc is a long video
RlDZgZvIQ5U is a long video


### Integrating Shorts classification with Music classification

In [191]:
BATCH_SIZE = 50

def create_video_batches(dataframe):
    combined_video_ids_batch = []

    for start_index in range(0, len(dataframe), BATCH_SIZE):
        video_ids_batch = dataframe["video_ids"][
            start_index:start_index + BATCH_SIZE
        ]

        combined_video_ids = ",".join(video_ids_batch)
        combined_video_ids_batch.append(combined_video_ids)

    return combined_video_ids_batch

def check_music_candidates(music_candidate_ids):
    true_music_candidates = []

    for music_candidate in music_candidate_ids:
        if is_short(music_candidate.get("video_ID")):
            print(f"{music_candidate.get("video_title")} is a short video")

        else:
            #print(f"{music_candidate.get("video_title")} is a long-form music video")
            true_music_candidates.append(music_candidate)

    return true_music_candidates

video_batches = create_video_batches(records_dataframe)
music_candidates = []

for combined_video_ids_string in video_batches:

    # API request...
    watch_history_params = {
        "part": "snippet,contentDetails",
        "id": combined_video_ids_string,
        "key": api_key
    }

    watch_history_response = requests.get(url, params=watch_history_params)
    watch_history_data = watch_history_response.json()

    for item in watch_history_data["items"]:
        snippet = item["snippet"]
        content = item["contentDetails"]

        video_id = item["id"]
        video_title = snippet["title"]
        video_category = snippet["categoryId"]
        channel_name = snippet["channelTitle"]
        channel_id = snippet["channelId"]

        if video_category == "10":
            music_candidate_entry = {
                "video_title": video_title,
                "video_ID": video_id,
                "channel_name": channel_name,
                "channel_id": channel_id,
            }
            music_candidates.append(music_candidate_entry)

actual_candidates = check_music_candidates(music_candidates)

print(f"{len(actual_candidates)} music candidates found")
for candidate in actual_candidates:
    print(f"{candidate.get("video_title")}")



Why Eminem HATES doing commercials 🤬😳 is a short video
PT.2: Driving in my Car (Asgore Runs Over Dess) but heavenly. Animation by  @xmikamarble ​ is a short video
Natural selection 🥀 is a short video
Bro MIGHT be the smartest ✌️ is a short video
Wonder what his barber does for a living 🥀 is a short video
Historically accurate tho 🙏 is a short video
They just took a hydration break 🥀 is a short video
Teach moving like John Wick ✌️ is a short video
He sounded so mad at chase 😭 is a short video
Bro understood the consequences ✌️ is a short video
"Coming early for tomorrow?" 💔 is a short video
First guy deserves an Oscar 😭 is a short video
"Half day huh?" 💔 is a short video
Bro picked the pro kids 🥀 is a short video
fat little chud teto cover #重音テト #synthesizerv is a short video
He noticed the actor’s lines slowed down on TV and kidnapped him for diagnosis#house #fyp is a short video
”You got any games on your phone?” 🥀 is a short video
Why did you steal my stuff #MD is a short video
Loki 

A really good passthrough, we caught 32 shorts from our list alone, which is great, of course, there are a still a bunch of outliers, but now we'll use Official channels and keywords to further refine that

## Official Channel/Topic Classification

### Using Youtube's API to request Channel metadata

Firstly, we'll need to look at the metadata on the channels from the video, we're inspecting

In [179]:
music_video_id = records_dataframe.loc[168, "video_ids"]
music_video_title = records_dataframe.loc[168,"title"]

print(f'The video: {music_video_title}, has the video id: {music_video_id}')

channel_url = "https://www.googleapis.com/youtube/v3/channels"

music_params = {
    "part": "snippet,contentDetails",
    "id": music_video_id,
    "key": api_key
}

music_video_response = requests.get(url, params=music_params)

music_video_data = music_video_response.json()
non_music_video_data = non_music_video_response.json()


mv_snippet = mv_item["snippet"]
mv_item = music_video_data["items"][0]
SIO_channel_name = mv_snippet["channelTitle"]
SIO_channel_id = mv_item["snippet"]["channelId"]

print(f"The Channel ID for {SIO_channel_name} who published {mv_snippet["title"]} is {SIO_channel_id}")

channel_params = {
    "part": "snippet,contentDetails,statistics,topicDetails",
    "id": SIO_channel_id,
    "key": api_key
}

channel_response = requests.get(channel_url, params=channel_params)

print(channel_response.status_code)

channel_data = channel_response.json()
channel_item = channel_data["items"][0]

#print(channel_item)


The video: Why Do I (with Hatsune Miku), has the video id: HMFNXgu7L3Q
The Channel ID for Set It Off who published Why Do I (with Hatsune Miku) is UCB1bJqKXS3lpy8HdW0P1zrQ
200
{'topicIds': ['/m/064t9', '/m/04rlf', '/m/06by7', '/m/02lkt'], 'topicCategories': ['https://en.wikipedia.org/wiki/Pop_music', 'https://en.wikipedia.org/wiki/Music', 'https://en.wikipedia.org/wiki/Rock_music', 'https://en.wikipedia.org/wiki/Electronic_music']}


In [178]:
print(channel_item.get("topicDetails"))

{'topicIds': ['/m/02lkt', '/m/064t9', '/m/04rlf', '/m/06by7'], 'topicCategories': ['https://en.wikipedia.org/wiki/Electronic_music', 'https://en.wikipedia.org/wiki/Pop_music', 'https://en.wikipedia.org/wiki/Music', 'https://en.wikipedia.org/wiki/Rock_music']}


In [189]:
non_official_music_channel_handle = "@KoganeMusic"
topic_channel_id = "UCJDvRiWvolEhRLPAVoOzGTA"
non_music_channel_id = "UCiZVMOinTQGb8HQu53VbV4Q"

def get_channel_id_from_handle(handle):
    handle_channel_params = {
        "part": "snippet",
        "forHandle": handle,
        "key": api_key
    }

    handle_channel_response = requests.get(channel_url, params=handle_channel_params)
    handle_channel_data = handle_channel_response.json()

    #print(handle_channel_data)
    snippet = handle_channel_data["items"][0]

    handle_channel_id = snippet["id"]
    #print(handle_channel_id)
    return handle_channel_id

NOM_channel_params = {
    "part": "snippet, topicDetails",
    "id": get_channel_id_from_handle(non_official_music_channel_handle),
    "key": api_key
}

topic_channel_params = {
    "part": "snippet, topicDetails",
    "id": topic_channel_id,
    "key": api_key
}

non_music_channel_params = {
    "part": "snippet, topicDetails",
    "id": non_music_channel_id,
    "key": api_key
}

NOM_channel_response = requests.get(channel_url, params=NOM_channel_params)
topic_channel_response = requests.get(channel_url, params=topic_channel_params)
non_music_channel_response = requests.get(channel_url, params=non_music_channel_params)

channel_data = channel_response.json()
NOM_channel_data = NOM_channel_response.json()
topic_data = topic_channel_response.json()
non_music_data = non_music_channel_response.json()

NOM_channel_item = NOM_channel_data["items"][0]
topic_channel_item = topic_data["items"][0]
non_music_item = non_music_data["items"][0]


Ok now we want to try the same for 3 other items:
A. Official artist
   Set It Off / Ado / YOASOBI

B. Unofficial but definitely music
   one of your cover or lyric channels

C. Definitely non-music
   any normal gaming/commentary channel

In [190]:
print(f"Topic details for Set it Off (Official Music Channel)"
      f"\n {channel_item.get("topicDetails")}")

print(f"Topic details for Kogane Music (Non Official Music Channel)"
      f"\n {NOM_channel_item.get("topicDetails")}")

print(f"Topic details for Lyn (Topic Music Channel)"
      f"\n {topic_channel_item.get("topicDetails")}")

print(f"Topic details for Little Gamer's Playground (Non Music Channel)"
      f"\n {non_music_item.get("topicDetails")}")


Topic details for Set it Off (Official Music Channel)
 {'topicIds': ['/m/064t9', '/m/04rlf', '/m/06by7', '/m/02lkt'], 'topicCategories': ['https://en.wikipedia.org/wiki/Pop_music', 'https://en.wikipedia.org/wiki/Music', 'https://en.wikipedia.org/wiki/Rock_music', 'https://en.wikipedia.org/wiki/Electronic_music']}
Topic details for Kogane Music (Non Official Music Channel)
 {'topicIds': ['/m/028sqc', '/m/04rlf'], 'topicCategories': ['https://en.wikipedia.org/wiki/Music_of_Asia', 'https://en.wikipedia.org/wiki/Music']}
Topic details for Lyn (Topic Music Channel)
 {'topicIds': ['/m/0bzvm2', '/m/028sqc', '/m/04rlf', '/m/064t9'], 'topicCategories': ['https://en.wikipedia.org/wiki/Video_game_culture', 'https://en.wikipedia.org/wiki/Music_of_Asia', 'https://en.wikipedia.org/wiki/Music', 'https://en.wikipedia.org/wiki/Pop_music']}
Topic details for Little Gamer's Playground (Non Music Channel)
 {'topicIds': ['/m/02ntfj', '/m/0bzvm2', '/m/025zzc', '/m/0403l3g'], 'topicCategories': ['https://en.

Interestingly, all 3 music channels contain the same topic ID: "/m/04rlf" which we can assume is the one needed for music, we can use this to classify as channel as a music channel for our filter

### Integrating Music Channel Classification with other classifications

In [193]:
def create_channel_batches(channel_ids):
    channel_ids = list(channel_ids)
    combined_channel_ids_batch = []

    for start_index in range(0, len(channel_ids), BATCH_SIZE):
        channel_ids_batch = channel_ids[start_index:start_index + BATCH_SIZE]
        combined_channel_ids = ",".join(channel_ids_batch)
        combined_channel_ids_batch.append(combined_channel_ids)

    return combined_channel_ids_batch

unique_channel_ids = set()
for candidate in actual_candidates:
    unique_channel_ids.add(candidate.get("channel_id"))

print(f"Number of videos at stage 2, {len(actual_candidates)}")
print(f"Number of unique channel ids, {len(unique_channel_ids)}")

channel_batches = create_channel_batches(unique_channel_ids)

music_channel_ids = set()
unknown_channel_ids = set()
stage_three_filtered_candidates = []

for channel_id_batch in channel_batches:
    channel_params = {
        "part": "snippet,topicDetails",
        "id": channel_id_batch,
        "key": api_key
    }

    channel_response = requests.get(
        channel_url,
        params=channel_params
    )

    channel_data = channel_response.json()

    for channel_item in channel_data["items"]:
        channel_id = channel_item["id"]
        channel_title = channel_item["snippet"]["title"]
        topic_details = channel_item.get("topicDetails")

        if topic_details and "/m/04rlf" in topic_details.get("topicIds", []):
            music_channel_ids.add(channel_id)
            print(f"{channel_title}: music channel")

        else:
            unknown_channel_ids.add(channel_id)
            print(f"{channel_title}: unknown channel")


print("Unique channels:", len(unique_channel_ids))
print("Music channels:", len(music_channel_ids))
print("Unknown channels:", len(unknown_channel_ids))




Number of videos at stage 2, 598
Number of unique channel ids, 177
Official髭男dism: music channel
Shingo Nishimura - Topic: music channel
sxndyalt: music channel
nui: music channel
DanFourts: music channel
Kailash Saire: music channel
InstrUmantal: unknown channel
ImagineDragonsVEVO: music channel
Tokai Teio (CV: Machico) - Topic: music channel
attouです: unknown channel
Kenta Higashiohji - Topic: music channel
青い子 / AOIKO: music channel
Mori Calliope - Topic: music channel
V I P E R (MMD): music channel
しゃみお - Shamisen player Shamio: music channel
Pageless - Topic: music channel
くじらofficial: music channel
tian nya: unknown channel
Release - Topic: unknown channel
Rainych Ran: music channel
Utada Zoey Ch. 歌田ぞおい: unknown channel
WanderingSky114: music channel
Zip: music channel
HIMEHINA Channel: music channel
Wildreamz: unknown channel
MyaMusical: music channel
TAKU INOUE: music channel
Ashvale: music channel
Alekirser Live: unknown channel
🐈Kogane Music: music channel
Talon: unknown chann

## Keyword classification

In [168]:
STRONG_MUSIC_KEYWORDS = [
    "official audio",
    "music video",
    "lyric video",
    "lyrics",
    "cover",
    "remix",
    "instrumental",
    "full song",
    "歌ってみた",
]

WEAK_MUSIC_KEYWORDS = [
    "song",
    "ost",
]

STRONG_NON_MUSIC_KEYWORDS = [
    "sound effect",
    "sfx",
    "asmr",
]

WEAK_NON_MUSIC_KEYWORDS = [
    "reaction",
    "movie clip",
    "tv clip",
    "scene",
    "funny moments",
]

def find_keyword_matches(title, keywords):
    keywords_found = []

    for keyword in keywords:
        if keyword in title.lower():
            keywords_found.append(keyword)

    return keywords_found

def check_candidates_for_keywords(music_candidates):
    for candidate in music_candidates:
        strong_positive_evidence = find_keyword_matches(candidate.get("video_title"), STRONG_MUSIC_KEYWORDS)
        weak_positive_evidence = find_keyword_matches(candidate.get("video_title"), WEAK_MUSIC_KEYWORDS)
        strong_negative_evidence = find_keyword_matches(candidate.get("video_title"), STRONG_NON_MUSIC_KEYWORDS)
        weak_negative_evidence = find_keyword_matches(candidate.get("video_title"), WEAK_NON_MUSIC_KEYWORDS)

        print(f"Strong music keywords in {candidate.get('video_title')}:  {strong_positive_evidence}")
        print(f"Weak music keywords in {candidate.get('video_title')}:  {weak_positive_evidence}")
        print(f"Strong non music keywords in {candidate.get('video_title')}:  {strong_negative_evidence}")
        print(f"weak non music keywords in {candidate.get('video_title')}: {weak_negative_evidence}\n")

check_candidates_for_keywords(actual_candidates)

Strong music keywords in Unknown Mother Goose (wowaka) ♡ English Cover【rachie】アンノウン・マザーグース:  ['cover']
Weak music keywords in Unknown Mother Goose (wowaka) ♡ English Cover【rachie】アンノウン・マザーグース:  []
Strong non music keywords in Unknown Mother Goose (wowaka) ♡ English Cover【rachie】アンノウン・マザーグース:  []
weak non music keywords in Unknown Mother Goose (wowaka) ♡ English Cover【rachie】アンノウン・マザーグース: []

Strong music keywords in Aldnoah.Zero - "aLIEz" (FULL Ending) | ENGLISH ver | AmaLee:  []
Weak music keywords in Aldnoah.Zero - "aLIEz" (FULL Ending) | ENGLISH ver | AmaLee:  []
Strong non music keywords in Aldnoah.Zero - "aLIEz" (FULL Ending) | ENGLISH ver | AmaLee:  []
weak non music keywords in Aldnoah.Zero - "aLIEz" (FULL Ending) | ENGLISH ver | AmaLee: []

Strong music keywords in "Violet Snow" (Violet Evergarden) 🍓 Cover by Lizz Robinett ft. The Goatee:  ['cover']
Weak music keywords in "Violet Snow" (Violet Evergarden) 🍓 Cover by Lizz Robinett ft. The Goatee:  []
Strong non music keywords in

I feel like this is a good way to find "non-official" videos like lyrics or covers, but it i feel like our keywords are being a bit restrictive, maybe instead of using them to flag a video as being music, we use it as the opposite, we use this to define what isn't music